# DS 208 · Week 10 — Text processing and regular expressions
## From string cleaning to pattern matching

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 7 — Patterns on these slides run in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 9 — Three methods for cleaning text


**Exercise.**

1. Predict the three printed strings, spaces included.
2. Run the cell.
3. Change the last line to s.strip().replace(" City", ""). What prints now?


In [ ]:
s = "  Cebu City  "

print(repr(s.strip()))
print(repr(s.strip().lower()))
print(repr(s.replace("City", "")))

### Slide 10 — The flood descriptions, loaded from the course API


Expected output:

```
(34079, 15) 0
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
df   = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))

d = df.description          # 34,079 descriptions, written by many people
print(df.shape, d.isna().sum())

### Slide 11 — Free text written by many authors


Expected output:

```
CONSTRUCTION 26552
REHABILITATION 6387
IMPROVEMENT 2791
REPAIR 1010
```


In [ ]:
for kw in ["CONSTRUCTION", "REHABILITATION",
           "IMPROVEMENT", "REPAIR"]:
    print(kw, d.str.contains(kw, na=False).sum())

### Slide 12 — split and join


Expected output:

```
['Cebu', 'Davao', 'Iloilo']
a, b, c
```


In [ ]:
print("Cebu,Davao,Iloilo".split(","))

print(", ".join(["a", "b", "c"]))

### Slide 15 — Searching for a shape of text


**Exercise.**

1. Predict both printed results.
2. Run the cell.
3. Change the text to "12 cats 4 dogs" and the pattern to r"\d". What does findall return?


In [ ]:
import re

m = re.search(r"\d+", "order 42")
print(m.group())

print(re.findall(r"\d+", "3 cats 4 dogs"))

### Slide 16 — A plain keyword count overstates by 393


Expected output:

```
26552
26159
```


In [ ]:
# substring: also catches RECONSTRUCTION
print(d.str.contains("CONSTRUCTION", na=False).sum())

# word boundary: the real count
print(d.str.contains(r"\bCONSTRUCTION\b", na=False).sum())

### Slide 20 — Error: a pattern built from the first rows


Expected output:

```
3311
0.097
```


In [ ]:
pat = r"^(\d{2})([A-Z]{1,2})(\d{5})$"
ex = df.contract_id.str.extract(pat)

print(ex[0].notna().sum())
print(round(ex[0].notna().mean(), 3))
# and NOT ONE error was raised

### Slide 22 — Flexible quantifiers for variable parts


Expected output:

```
1.0
['14', '15', '16'] ... ['25', '26']
```


In [ ]:
pat = r"^(\d{2})([A-Z]+)(\d+)$"
ex = df.contract_id.str.extract(pat)
print(ex[0].notna().mean())

# the year prefix decodes too
years = sorted(ex[0].unique())
print(years[:3], "...", years[-2:])

### Slide 23 — 1,266 IDs disagree with their year column


Expected output:

```
1266 0.037
contract_id  year
325     20BH0131  2019
856     22MF0066  2021
1720    20CH0073  2019
```


In [ ]:
# ex: the decoded IDs from the previous slide
prefix_year = "20" + ex[0]
bad = df[prefix_year != df.year.astype(str)]

print(len(bad), round(len(bad) / len(df), 3))
print(bad[["contract_id", "year"]].head(3))

### Slide 28 — Parentheses capture parts of a match


Expected output:

```
2024
08
```


In [ ]:
import re

m = re.search(r"(\d{4})-(\d{2})", "2024-08")
print(m.group(1))   # the year
print(m.group(2))   # the month

### Slide 29 — Greedy and lazy matching on two bracket pairs


**Exercise.**

1. Predict what each of the two patterns returns.
2. Run the cell.
3. Add " (PHASE II)" to the end of t. What does the greedy pattern return now?


In [ ]:
import re

t = "CONSTRUCTION OF OCAÑA RIVER (GABIONS) (UPSTREAM), CARCAR CITY, CEBU"

# greedy: runs to the LAST )
print(re.search(r"\((.*)\)", t).group(1))

# lazy: stops at the FIRST )
print(re.search(r"\((.*?)\)", t).group(1))

### Slide 30 — Retrieving every match with findall


Expected output:

```
['GABIONS', 'UPSTREAM']
0
match
0 0       GABIONS
1      UPSTREAM
1 0      PHASE II
```


In [ ]:
import re

t = "CONSTRUCTION OF OCAÑA RIVER (GABIONS) (UPSTREAM), CARCAR CITY, CEBU"
print(re.findall(r"\((.*?)\)", t))

# a whole column: one row per match
s = pd.Series([t, "DIKE (PHASE II)"])
print(s.str.extractall(r"\((.*?)\)"))

### Slide 32 — .str applies a pattern to every row


Expected output:

```
['Ana', 'Cris']
0
0  2024
1  2023
2  2025
```


In [ ]:
contacts = pd.DataFrame({
    "name":  ["Ana", "Ben", "Cris"],
    "phone": ["09171234567", "0917-123", "09228889999"],
    "code":  ["FC-2024-031", "FC-2023-118", "FC-2025-007"]})

valid = contacts["phone"].str.contains(r"^09\d{9}$")
print(contacts[valid].name.tolist())          # rows with a valid mobile

print(contacts["code"].str.extract(r"(\d{4})"))   # the year, as a column

### Slide 34 — A place-name pattern that matches 36%


Expected output:

```
12400 0.364
description
AGNO RIVER         140
ANGAT RIVER        131
RIO CHICO RIVER    105
Name: count, dtype: int64
```


In [ ]:
places = d.str.extract(
    r"\b(?:ALONG|IN)\s+([A-Z][A-Z\s\.]{3,40}?)(?:,|\s+\()",
    expand=False)

print(places.notna().sum(), round(places.notna().mean(), 3))
print(places.dropna().str.strip().value_counts().head(3))

### Slide 38 — Compiling for readability rather than speed


Week 5's timing helper, used by the next cell:


In [ ]:
import timeit

def time_ms(f):
    """Average milliseconds per call of f()."""
    calls, seconds = timeit.Timer(f).autorange()
    return seconds / calls * 1000

Timings measured on the lecture laptop; yours will differ, so compare the ratios:

```
25.6 ms, 22.0 ms, 1.16x
```


In [ ]:
# time_ms(f) is week 5's timeit helper
descriptions = df.description.tolist()
pat = r"\bRIVER\b"

def plain():
    for s in descriptions:
        re.search(pat, s)

rx = re.compile(pat)
def compiled():
    for s in descriptions:
        rx.search(s)

a, b = time_ms(plain), time_ms(compiled)
print(f"{a:.1f} ms, {b:.1f} ms, {a / b:.2f}x")

### Slide 44 — Extracting every mobile number from a message


**Exercise.**

1. Predict the list that prints.
2. Run the cell.
3. Add , 0917 123 4567 (with spaces) to the text. Is it found?


In [ ]:
import re

text = "Call 09171234567 or 09228889999"
print(re.findall(r"09\d{9}", text))

### Slide 45 — Using patterns to rewrite text


**Exercise.**

1. Predict both cleaned strings.
2. Run the cell.
3. Delete .strip() from the first line. What changes?


In [ ]:
import re

s = "CONST.  OF   FLOOD CONTROL   STRUCTURE,   CEBU "
print(re.sub(r"\s+", " ", s).strip())

# reference a group with \1
print(re.sub(r"\s+([,.])", r"\1", "STRUCTURE , CEBU ."))

### Slide 46 — Naming groups instead of numbering them


Expected output:

```
['yr', 'office', 'seq']
yr office    seq
0  22      F  00136
1  18     BI   0029
2  19     CD   0029
```


In [ ]:
pat = r"^(?P<yr>\d{2})(?P<office>[A-Z]+)(?P<seq>\d+)$"
ex = df.contract_id.str.extract(pat)

print(list(ex.columns))
print(ex.head(3))

### Slide 47 — Flags that change the behaviour of a whole pattern


Expected output:

```
<re.Match object; span=(13, 17), match='CEBU'>
('22', 'F', '00136')
```


In [ ]:
import re

print(re.search(r"cebu", "CARCAR CITY, CEBU", re.IGNORECASE))

# VERBOSE: whitespace and comments inside the pattern
rx = re.compile(r"""
    ^(\d{2})      # 2-digit year
    ([A-Z]+)      # office code
    (\d+)$        # sequence
""", re.VERBOSE)
print(rx.match("22F00136").groups())

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
